In [ ]:
# ============================================================
# Descriptive Statistics Report Generator
# ============================================================
# Purpose:
#   Generate descriptive statistics for HRS variables across
#   multiple waves with detailed breakdowns.
# ============================================================

from pyspark.sql import functions as F
import pandas as pd

# ============================================================
# INPUT PARAMETERS
# ============================================================
catalog_name = "staging_catalog"          # UC catalog name
schema_name = "slv_cdm_hrs_stg"           # UC schema name
table_name = "hrs_demographics"           # Table to analyze
variable_name = "agey_e"                  # Variable/column to analyze
wave_numbers = [1, 2, 3, 4, 5, 6, 7, 8]   # List of waves to include

# ============================================================
# DATA RETRIEVAL
# ============================================================

# Construct fully qualified table name
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

print(f"Analyzing variable: {variable_name}")
print(f"From table: {full_table_name}")
print(f"Waves: {wave_numbers}")
print("="*60)

# Read the table
df = spark.table(full_table_name)

# Filter for specified waves and ensure variable exists
if "wave" in df.columns and variable_name in df.columns:
    df_filtered = df.filter(F.col("wave").isin(wave_numbers))
else:
    raise ValueError(f"Required columns missing. Table must have 'wave' and '{variable_name}' columns.")

# ============================================================
# CALCULATE STATISTICS BY WAVE
# ============================================================

stats_by_wave = df_filtered.groupBy("wave").agg(
    F.count(F.col(variable_name)).alias("N"),
    F.min(F.col(variable_name)).alias("Min"),
    F.max(F.col(variable_name)).alias("Max"),
    F.mean(F.col(variable_name)).alias("Mean"),
    F.stddev(F.col(variable_name)).alias("STD")
).orderBy("wave")

# Convert to pandas for better display formatting
stats_df = stats_by_wave.toPandas()

# ============================================================
# CALCULATE OVERALL STATISTICS (ALL WAVES COMBINED)
# ============================================================

overall_stats = df_filtered.agg(
    F.count(F.col(variable_name)).alias("N"),
    F.min(F.col(variable_name)).alias("Min"),
    F.max(F.col(variable_name)).alias("Max"),
    F.mean(F.col(variable_name)).alias("Mean"),
    F.stddev(F.col(variable_name)).alias("STD")
).toPandas()

# Add wave column for overall stats
overall_stats.insert(0, "wave", "Overall")

# ============================================================
# COMBINE AND FORMAT RESULTS
# ============================================================

# Combine wave-specific and overall statistics
final_stats = pd.concat([stats_df, overall_stats], ignore_index=True)

# Format numeric columns to 2 decimal places
for col in ["Min", "Max", "Mean", "STD"]:
    final_stats[col] = final_stats[col].round(2)

# Rename wave column for clarity
final_stats.rename(columns={"wave": "Wave"}, inplace=True)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print(f"\nDescriptive Statistics for: {variable_name}")
print(f"Table: {full_table_name}")
print("="*60)

display(final_stats)